In [ ]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path("../..").resolve()))
from configs.config import JSON_CHUNKS_DIR

# --- Papermill parameters (overwritten at runtime) ---
input_file      = str(JSON_CHUNKS_DIR / "rag_chunks_all.jsonl")
output_file     = str(JSON_CHUNKS_DIR / "rag_chunks_split_langchain.jsonl")
chunk_size      = 1500
chunk_overlap   = 200
experiment_name = "baseline"
run_id          = "run_1"

In [ ]:
import mlflow

# Asegurar que estamos en el run correcto sin iniciar uno nuevo
if mlflow.active_run() is None and "run_id" in globals():
    mlflow.start_run(run_id=run_id)

In [ ]:
import os
import json
import mlflow
import time
from langchain.text_splitter import RecursiveCharacterTextSplitter

mlflow.log_params({
    "input_file":    input_file,
    "output_file":   output_file,
    "chunk_size":    chunk_size,
    "chunk_overlap": chunk_overlap,
})

os.makedirs(os.path.dirname(output_file), exist_ok=True)

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=chunk_size,
    chunk_overlap=chunk_overlap,
    length_function=len,
)

total_subchunks  = 0
total_char_count = 0
min_size = float("inf")
max_size = 0
num_source_chunks = 0
start_time = time.time()

with open(input_file, "r", encoding="utf-8") as fin, \
     open(output_file, "w", encoding="utf-8") as fout:
    for line in fin:
        record  = json.loads(line)
        doc_id  = record["doc_id"]
        page    = record["page"]
        content = record["content"]
        num_source_chunks += 1

        for i, sub_text in enumerate(text_splitter.split_text(content)):
            sub_len = len(sub_text)
            total_subchunks  += 1
            total_char_count += sub_len
            min_size = min(min_size, sub_len)
            max_size = max(max_size, sub_len)

            fout.write(json.dumps({
                "doc_id":   doc_id,
                "page":     page,
                "chunk_id": f"{doc_id}_p{page}_c{i}",
                "content":  sub_text,
                "source":   record.get("source", input_file),
            }, ensure_ascii=False) + "\n")

processing_time = time.time() - start_time
avg_size = total_char_count / total_subchunks if total_subchunks else 0

mlflow.log_metrics({
    "total_subchunks":    total_subchunks,
    "avg_chunk_size":     round(avg_size),
    "min_chunk_size":     min_size,
    "max_chunk_size":     max_size,
    "source_chunks":      num_source_chunks,
    "processing_time_s":  round(processing_time, 2),
})

print(f"{total_subchunks} sub-chunks written to: {output_file}")
print(f"avg size: {avg_size:.0f} chars | min: {min_size} | max: {max_size}")